# Скрит позволяет разбивать изображения на части для обучения c метками

In [14]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# Параметры разбиения
PATCH_SIZE = 640
OVERLAP = 10

DATA_DIR = "../../DataSet/DataFullHD/Valid/"
OUTPUT_DIR = "../../DataSet/DataFullHD/Split/Valid/" # куда


In [15]:
import os
import cv2

def split_image_and_labels(image_path, label_path, patch_size, overlap, output_dir):
    '''
        Разбитие изображений на фразменты
    '''
    image = cv2.imread(image_path)
    h, w, _ = image.shape
    step = patch_size - overlap

    os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels"), exist_ok=True)

    # Читаем метки (если файл пустой, то labels = [])
    labels = []
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:  # Фильтруем строки, где не 5 значений
                    labels.append([float(x) for x in parts])

    # Разрезаем изображение
    idx = 0
    for y in range(0, h, step):
        for x in range(0, w, step):
            x_end = min(x + patch_size, w)
            y_end = min(y + patch_size, h)
            patch = image[y:y_end, x:x_end]
            patch_h, patch_w, _ = patch.shape

            # Генерируем имя файла
            img_name = f"{os.path.splitext(os.path.basename(image_path))[0]}_{idx}.jpg"
            lbl_name = f"{os.path.splitext(os.path.basename(label_path))[0]}_{idx}.txt"

            cv2.imwrite(os.path.join(output_dir, "images", img_name), patch)

            # Собираем метки для этого патча
            patch_labels = []
            for label in labels:
                if len(label) == 5:  # Защита от некорректных данных
                    cls, x_center, y_center, width, height = label
                    x_abs = x_center * w
                    y_abs = y_center * h
                    box_w = width * w
                    box_h = height * h

                    # Проверяем, попадает ли объект в область
                    if x < x_abs < x_end and y < y_abs < y_end:
                        new_x_center = (x_abs - x) / patch_w
                        new_y_center = (y_abs - y) / patch_h
                        new_width = box_w / patch_w
                        new_height = box_h / patch_h
                        patch_labels.append([cls, new_x_center, new_y_center, new_width, new_height])

            # Сохраняем метки, если они есть
            label_output_path = os.path.join(output_dir, "labels", lbl_name)
            if patch_labels:  # Записываем только если есть метки
                with open(label_output_path, "w") as f:
                    for lbl in patch_labels:
                        f.write(" ".join(map(str, lbl)) + "\n")
            else:
                open(label_output_path, "w").close()

            idx += 1


In [16]:

# Обработка всего набора данных

for img_name in tqdm(os.listdir(os.path.join(DATA_DIR, "images"))):
    img_path = os.path.join(DATA_DIR, "images", img_name)
    label_path = os.path.join(DATA_DIR, "labels", f"{os.path.splitext(img_name)[0]}.txt")

    split_image_and_labels(
        image_path=img_path,
        label_path=label_path,
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        output_dir=OUTPUT_DIR,
    )

100%|██████████| 400/400 [00:34<00:00, 11.58it/s]


In [19]:
import os
import cv2

images_dir = '../../DataSet/DataFullHD/Split/Test/images/'
labels_dir = '../../DataSet/DataFullHD/Split/Test/labels/'

min_width = 200

for filename in os.listdir(images_dir):
    if not (filename.lower().endswith('.jpg') or filename.lower().endswith('.png') or filename.lower().endswith('.jpeg')):
        continue

    image_path = os.path.join(images_dir, filename)
    img = cv2.imread(image_path)

    if img is None:
        print(f"Не удалось прочитать {filename}, пропускаем.")
        continue

    height, width = img.shape[:2]

    if width < min_width:
        print(f"Удаляем {filename} с шириной {width}px")

        # Удаляем изображение
        os.remove(image_path)

        # Удаляем соответствующий label
        label_filename = os.path.splitext(filename)[0] + '.txt'
        label_path = os.path.join(labels_dir, label_filename)
        if os.path.exists(label_path):
            os.remove(label_path)
            print(f"Удалён label {label_filename}")
        else:
            print(f"Label для {filename} не найден")

Удаляем 0_3.jpg с шириной 30px
Удалён label 0_3.txt
Удаляем 0_7.jpg с шириной 30px
Удалён label 0_7.txt
Удаляем 100_3.jpg с шириной 30px
Удалён label 100_3.txt
Удаляем 100_7.jpg с шириной 30px
Удалён label 100_7.txt
Удаляем 101_3.jpg с шириной 30px
Удалён label 101_3.txt
Удаляем 101_7.jpg с шириной 30px
Удалён label 101_7.txt
Удаляем 102_3.jpg с шириной 30px
Удалён label 102_3.txt
Удаляем 102_7.jpg с шириной 30px
Удалён label 102_7.txt
Удаляем 103_3.jpg с шириной 30px
Удалён label 103_3.txt
Удаляем 103_7.jpg с шириной 30px
Удалён label 103_7.txt
Удаляем 104_3.jpg с шириной 30px
Удалён label 104_3.txt
Удаляем 104_7.jpg с шириной 30px
Удалён label 104_7.txt
Удаляем 105_3.jpg с шириной 30px
Удалён label 105_3.txt
Удаляем 105_7.jpg с шириной 30px
Удалён label 105_7.txt
Удаляем 106_3.jpg с шириной 30px
Удалён label 106_3.txt
Удаляем 106_7.jpg с шириной 30px
Удалён label 106_7.txt
Удаляем 107_3.jpg с шириной 30px
Удалён label 107_3.txt
Удаляем 107_7.jpg с шириной 30px
Удалён label 107_7.txt


In [1]:
# Если просто разрезать картинку
from PIL import Image
import os

def crop_image_to_squares(image_path, output_dir, crop_size=640):
    # Открываем изображение
    image = Image.open(image_path)
    width, height = image.size

    # Создаём выходную папку, если её нет
    os.makedirs(output_dir, exist_ok=True)

    count = 0
    for top in range(0, height, crop_size):
        for left in range(0, width, crop_size):
            # Определяем границы обрезки
            right = min(left + crop_size, width)
            bottom = min(top + crop_size, height)

            # Пропускаем куски меньше 640x640 (можно убрать, если нужно сохранить даже неполные)
            if right - left < crop_size or bottom - top < crop_size:
                continue

            # Вырезаем квадрат
            box = (left, top, right, bottom)
            cropped = image.crop(box)

            # Сохраняем с нумерацией
            cropped.save(os.path.join(output_dir, f"crop_{count}.png"))
            count += 1

    print(f"Готово! Сохранено {count} квадратов в '{output_dir}'")


crop_image_to_squares("11111.jpg", "2222.jpg")

Готово! Сохранено 2 квадратов в '2222.jpg'
